# 3.3 LCEL — LangChain Expression Language

## Playground Notebook

LCEL is LangChain's **declarative composition framework**. It lets you build complex LLM pipelines by chaining components with the pipe `|` operator.

| Topic | What You'll Learn |
|-------|-------------------|
| **The Pipe Operator** | Building chains with `prompt \| model \| parser` |
| **Runnable Protocol** | `invoke()`, `stream()`, `batch()` — the universal interface |
| **RunnablePassthrough** | Forwarding and enriching data through chains |
| **RunnableParallel** | Executing multiple branches simultaneously |
| **RunnableLambda** | Injecting custom Python functions into chains |
| **RunnableBranch** | Conditional routing (if/else for chains) |
| **Fallbacks & Retry** | Automatic error recovery and resilience |
| **Streaming & Batching** | First-class streaming and parallel batch processing |

Think of LCEL as the **plumbing system** — individual components are useful on their own, but LCEL connects them into a pipeline where data flows seamlessly and streaming, async, batching, and observability come for free.

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [ ]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [2]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch,
)
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready


---

## 1. The Runnable Protocol — Foundation of LCEL

Every component in LCEL implements the **Runnable protocol** — a universal interface that makes composition possible.

| Method | Description | Use Case |
|--------|-------------|----------|
| `invoke()` | Process single input synchronously | Standard request-response |
| `stream()` | Yield results incrementally | Real-time chat UIs |
| `batch()` | Process multiple inputs in parallel | Bulk processing |
| `ainvoke()` | Async invoke | High-concurrency servers |
| `astream()` | Async streaming | Async chat apps |

**Critical insight:** When you compose Runnables with `|`, the resulting chain is *itself* a Runnable. So you can `invoke()`, `stream()`, or `batch()` the entire chain.

### Input/Output Types

| Component | Input | Output |
|-----------|-------|--------|
| `ChatPromptTemplate` | `dict` (variables) | `ChatPromptValue` (messages) |
| `ChatModel` | messages | `AIMessage` |
| `StrOutputParser` | `AIMessage` | `str` |
| `JsonOutputParser` | `AIMessage` | `dict` |
| `RunnablePassthrough` | any | same as input |
| `RunnableLambda` | any | any (depends on function) |

### Experiment 1A: The Fundamental Chain Pattern

In [3]:
# The fundamental pattern: prompt | model | parser
#
# Data flow:
#   {variables} → ChatPromptTemplate → Messages → ChatModel → AIMessage → StrOutputParser → string

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical assistant."),
    ("human", "{question}")
])

parser = StrOutputParser()

# Build the chain with the pipe operator
chain = prompt | llm | parser

# The chain is itself a Runnable
print(f"Chain type: {type(chain).__name__}")

# invoke() — single input, full response
result = chain.invoke({"question": "What is LCEL in LangChain? 2 sentences."})

print(f"Result type: {type(result).__name__}")  # str, not AIMessage
display(Markdown(result))

Chain type: RunnableSequence
Result type: TextAccessor


LCEL stands for Language Chain Execution Layer, which is a component of the LangChain library that enables the execution of complex language tasks by breaking them down into smaller, more manageable sub-tasks. By chaining together these sub-tasks using LCEL, developers can build robust and scalable language models that can be used to perform a wide range of NLP tasks.

### Experiment 1B: The Runnable Protocol in Action — `invoke`, `stream`, `batch`

In [4]:
# The SAME chain supports all three modes automatically

# --- invoke: single input ---
print("=" * 50)
print("invoke() — Single input, full response")
print("=" * 50)
result = chain.invoke({"question": "What is the Runnable protocol? One sentence."})
print(result)

invoke() — Single input, full response
The Runnable Protocol is a communication protocol that defines how a process or thread can be executed, including the exchange of control messages between threads, to enable concurrent programming and parallel execution of tasks.


In [5]:
# --- stream: token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in chain.stream({"question": "Name 3 LCEL primitives. Brief."}):
    print(chunk, end="", flush=True)
print()

stream() — Tokens arrive as generated
Here are three LCE (Local Computation Engine) primitives:

1. **Assignment**: Assigns a value to a variable.
2. **Negation**: Negates a boolean expression or a variable's value.
3. **Conjunction** (AND): Combines two boolean expressions into one.


In [6]:
# --- batch: multiple inputs in parallel ---
print("=" * 50)
print("batch() — Process multiple inputs at once")
print("=" * 50)

questions = [
    {"question": "What is RunnablePassthrough? One sentence."},
    {"question": "What is RunnableParallel? One sentence."},
    {"question": "What is RunnableLambda? One sentence."},
]

start = time.time()
results = chain.batch(questions)
elapsed = time.time() - start

for q, r in zip(questions, results):
    print(f"Q: {q['question']}")
    print(f"A: {r}\n")

print(f"Processed {len(questions)} inputs in {elapsed:.2f}s (parallel!)")

batch() — Process multiple inputs at once
Q: What is RunnablePassthrough? One sentence.
A: RunnablePassthrough is a technique used to enable running pre-trained models, such as Sentence-BERT or BERT, directly on input text without requiring additional fine-tuning or preprocessing steps.

Q: What is RunnableParallel? One sentence.
A: RunnableParallel is a lightweight, open-source library for parallelizing computationally expensive tasks in Python, allowing developers to easily scale their code to multiple CPU cores or even distributed computing environments.

Q: What is RunnableLambda? One sentence.
A: RunnableLambda is an open-source Python library that provides a simple way to create, train, and deploy machine learning models using popular deep learning frameworks like Hugging Face Transformers and PyTorch, with a focus on text classification and natural language processing tasks.

Processed 3 inputs in 2.38s (parallel!)


---

## 2. Extending the Chain — Adding More Components

Each additional component in the pipe adds a processing step. The chain remains a single Runnable.

```
chain = prompt | model | parser | post_processor | formatter
```

### Experiment 2A: Extending with Post-Processing

In [7]:
# Add a post-processing step using RunnableLambda
def add_disclaimer(text):
    return f"{text}\n\n--- Generated by {MODEL} via LCEL ---"

# Extended chain: prompt → model → parse → post-process
extended_chain = prompt | llm | StrOutputParser() | RunnableLambda(add_disclaimer)

result = extended_chain.invoke({"question": "What is LangChain? One sentence."})
print(result)

LangChain is an open-source Python library that provides a flexible and modular framework for building custom natural language processing (NLP) pipelines, making it easier to integrate various NLP components and tools into workflows.

--- Generated by llama3.2:3b via LCEL ---


### Experiment 2B: Chain of Chains — Composing Multiple Chains

In [8]:
# Chain 1: Generate a technical explanation
explain_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a technical writer. 2-3 sentences."),
        ("human", "Explain: {topic}")
    ])
    | llm
    | StrOutputParser()
)

# Chain 2: Simplify for a child
simplify_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Simplify this for a 10-year-old. Use an analogy. 2 sentences."),
        ("human", "{text}")
    ])
    | llm
    | StrOutputParser()
)

# Run in sequence — output of chain 1 feeds into chain 2
topic = "LCEL pipe operator"

print("Step 1 — Technical Explanation:")
print("=" * 50)
explanation = explain_chain.invoke({"topic": topic})
display(Markdown(explanation))

print("\nStep 2 — Simplified:")
print("=" * 50)
simple = simplify_chain.invoke({"text": explanation})
display(Markdown(simple))

Step 1 — Technical Explanation:


The `LCElipsisPipe` operator is a new addition to the PyTorch library, introduced in version 1.14.0. It's a type of tensor manipulation operation that allows you to "elide" (or remove) trailing zeros from tensors.

**What does it do?**

When working with tensors, sometimes trailing zeros are present due to rounding or numerical computations. The `LCElipsisPipe` operator removes these trailing zeros from the tensor, effectively "eliding" them.

**Example usage:**
```python
import torch

# Create a tensor with some trailing zeros
x = torch.tensor([1.23456789012345678, 2.3456789012345678])

# Use LCElipsisPipe to remove trailing zeros
y = x.lce_()

print(y)  # Output: [1.23456789 2.3456789]
```
As you can see, the `LCElipsisPipe` operator has removed the trailing zeros from the tensor.

**How does it work?**

The `LCElipsisPipe` operator uses a technique called "elision" to remove trailing zeros. It works by iterating over the elements of the tensor and checking if each element is non-zero. If an element is zero, it's simply skipped. The resulting tensor has no trailing zeros.

**Why is it useful?**

The `LCElipsisPipe` operator can be useful in various situations:

* **Reducing numerical noise**: In some cases, trailing zeros can lead to numerical instability or errors. Removing these trailing zeros can help improve the accuracy of computations.
* **Simplifying numerical representations**: Trailing zeros can make numerical representations more cumbersome and harder to work with. Removing them can simplify the representation and make it easier to manipulate.

**Other uses:**

The `LCElipsisPipe` operator has other use cases, such as:

* **Handling floating-point precision issues**: In some cases, trailing zeros can be due to floating-point precision errors. The `LCElipsisPipe` operator can help remove these errors.
* **Simplifying tensor representations in machine learning models**: Removing trailing zeros from tensors can simplify the representation and improve model performance.

Overall, the `LCElipsisPipe` operator is a useful tool for working with tensors in PyTorch. It's designed to simplify numerical computations by removing trailing zeros and improving the accuracy of results.


Step 2 — Simplified:


Imagine you're baking a cake and you need to measure out ingredients precisely. You want to make sure that your measurements are accurate, so you don't end up with too much or too little of something.

In computer science, when we work with numbers, sometimes we get "trailing zeros" - tiny extra bits at the end of a number that can affect how it's used in calculations. These trailing zeros can be like tiny mistakes in our baking recipe!

The `LCElipsisPipe` operator is like a special tool that helps us clean up these trailing zeros. It takes a number (or a group of numbers, like a tensor) and removes any trailing zeros, so we're left with just the important bits.

For example, if we have a number like 1.23456789, the `LCElipsisPipe` operator would remove the 9 at the end, leaving us with 1.2345678. This helps us avoid tiny mistakes and makes our calculations more accurate.

So, why is it useful? Well, when we're working with numbers in computers, these trailing zeros can sometimes cause problems or lead to errors. By removing them, the `LCElipsisPipe` operator helps us get more precise results and makes our computations easier and more reliable.

---

## 3. LCEL Primitive 1: RunnablePassthrough — Pass Data Through

`RunnablePassthrough` passes its input through **unchanged** to the next component. Seems trivial, but it's essential for preserving data when you need it downstream.

**`RunnablePassthrough.assign()`** — keeps the original input AND adds new keys. Crucial for enriching data as it flows.

### Experiment 3A: RunnablePassthrough Basics

In [9]:
# RunnablePassthrough — input passes through unchanged
passthrough = RunnablePassthrough()

result = passthrough.invoke("Hello, I pass right through!")
print(f"Input:  'Hello, I pass right through!'")
print(f"Output: '{result}'")
print(f"Same?   {result == 'Hello, I pass right through!'}")

Input:  'Hello, I pass right through!'
Output: 'Hello, I pass right through!'
Same?   True


### Experiment 3B: RunnablePassthrough.assign() — Enrich Data

In [10]:
# assign() keeps original input AND adds new computed keys

# Simulate: given a question, add a "word_count" and "uppercased" field
enricher = RunnablePassthrough.assign(
    word_count=RunnableLambda(lambda x: len(x["question"].split())),
    uppercased=RunnableLambda(lambda x: x["question"].upper())
)

result = enricher.invoke({"question": "What is LCEL in LangChain?"})

print("Input:  {'question': 'What is LCEL in LangChain?'}")
print(f"Output: {result}")
print()
print("Original 'question' is preserved, plus new keys added!")

Input:  {'question': 'What is LCEL in LangChain?'}
Output: {'question': 'What is LCEL in LangChain?', 'word_count': 5, 'uppercased': 'WHAT IS LCEL IN LANGCHAIN?'}

Original 'question' is preserved, plus new keys added!


### Experiment 3C: assign() in a Real Chain — Adding Context

In [11]:
# Practical use: enrich input with LLM-generated context before the main prompt

# Step 1: Generate keywords from the question
keyword_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Extract 3 keywords from this question. Comma-separated, no extra text."),
        ("human", "{question}")
    ])
    | llm
    | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add keywords
enriched_chain = RunnablePassthrough.assign(
    keywords=keyword_chain
)

# Step 3: Use both in the final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question. Focus on these keywords: {keywords}"),
    ("human", "{question}")
])

full_chain = enriched_chain | final_prompt | llm | StrOutputParser()

result = full_chain.invoke({"question": "How does LCEL handle streaming in chains?"})

print("Result:")
display(Markdown(result))

Result:


LCE (Loss Component Estimation) is a method used to efficiently process large amounts of text data, such as sentence transformers, by handling streaming in chains.

**Streaming**: When processing a new chain, the model starts from the beginning of the chain and streams one sentence at a time. This means that instead of loading the entire chain into memory, the model processes each sentence individually as it is streamed in.

**Chunking**: Each input sentence is chunked into smaller sub-sentences or "tokens" that can be processed independently. These tokens are typically shorter than words and are used to reduce the computational complexity of processing individual sentences.

**Loss calculation**: For each token in the chunk, the model calculates a loss component using a combination of the previous tokens in the chain and the current token. The loss components are calculated based on a pre-defined scoring function that takes into account the relationships between the tokens.

Here is a high-level overview of how LCE handles streaming in chains:

1. **Pre-processing**: Each sentence is chunked into smaller tokens, such as subwords or words.
2. **Streaming**: One token at a time, the model processes each token from the first sentence of the chain.
3. **Loss calculation**: For each token, the model calculates a loss component based on its relationships with previous tokens in the chain using the pre-defined scoring function.
4. **Weighted sum**: The loss components for all tokens in the chunk are combined using a weighted sum to produce an overall loss value for the chunk.
5. **Chain-wise aggregation**: The loss values for each chunk in the chain are aggregated using a combination of the chunk losses (e.g., mean or sum) to produce an overall chain-level loss.

**Key benefits**:

* **Efficient processing**: LCE allows for efficient processing of large amounts of text data by reducing the computational complexity of processing individual sentences.
* **Scalability**: By handling streaming in chains, LCE can handle very large datasets and is suitable for applications where scalability is important.
* **Flexibility**: LCE can be applied to various models, including sentence transformers, and can be modified to suit specific use cases.

**Chains**: In the context of sentence transformers, a chain refers to a contiguous sequence of sentences or documents that are stored together as a single unit. The model processes each chain from beginning to end, one chunk at a time, using the LCE method. This allows for efficient processing and storage of large amounts of text data.

By handling streaming in chains, LCE enables sentence transformers to process large datasets efficiently while maintaining high accuracy and scalability.

---

## 4. LCEL Primitive 2: RunnableParallel — Execute Branches Simultaneously

`RunnableParallel` takes a single input and routes it to **multiple Runnables that execute in parallel**. Results are collected into a dictionary.

```
Input
  ├──→ Branch A → result_a
  ├──→ Branch B → result_b
  └──→ Branch C → result_c
  ▼
{"a": result_a, "b": result_b, "c": result_c}
```

**Dictionary shorthand:** `{"key": runnable}` is automatically treated as `RunnableParallel`.

### Experiment 4A: Basic RunnableParallel

In [12]:
# Two branches process the same input in parallel

summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in one sentence."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

keywords_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List 3 keywords. Comma-separated only."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# RunnableParallel — both execute at the same time
parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain
)

start = time.time()
result = parallel_chain.invoke({"topic": "LangChain Expression Language (LCEL)"})
elapsed = time.time() - start

print(f"Completed in {elapsed:.2f}s (both branches ran in parallel!)\n")
print(f"Summary:  {result['summary']}")
print(f"Keywords: {result['keywords']}")

Completed in 16.30s (both branches ran in parallel!)

Summary:  **LangChain Expression Language (LCEL)** is a declarative language designed to enable data scientists, engineers, and researchers to easily build, deploy, and integrate large language models into various applications.

LCEL is built on top of the popular LangChain framework, which provides a set of tools for building and managing workflows that involve natural language processing (NLP) tasks. LCEL allows users to define complex NLP pipelines in a concise and readable way, using a syntax similar to Python's list comprehensions or SQL queries.

Some key features of LCEL include:

1. **Declarative syntax**: LCEL uses a declarative syntax that focuses on what the model should achieve, rather than how it should be implemented.
2. **Modular architecture**: LCEL pipelines can be broken down into smaller, independent modules that can be reused across different applications.
3. **Flexibility and extensibility**: LCEL allows users t

### Experiment 4B: Dictionary Shorthand (Most Common Pattern)

In [14]:
# Dictionary shorthand — cleaner syntax for RunnableParallel
# This is how it's used most often in real LCEL chains

analyze_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are given a topic with its summary and keywords.\n"
     "Summary: {summary}\n"
     "Keywords: {keywords}\n"
     "Write a brief analysis (2 sentences)."),
    ("human", "Analyze this topic.")
])

# Chain: parallel branches → combine into prompt → model → parse
full_chain = (
    {"summary": summary_chain, "keywords": keywords_chain}  # dict shorthand = RunnableParallel
    | analyze_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"topic": "The Runnable protocol in LangChain"})
display(Markdown(result))

**Topic Analysis: Sentence Transformers**

**Definition:** A type of deep learning model that transforms sentences into numerical vectors, capturing their semantic meaning.

**Key Concepts:**

1. **Sentence Embeddings**: The process of converting a sentence into a numerical vector representation.
2. **Multi-Layer Bidirectional Transformer Encoder**: The architecture used to learn sentence embeddings.
3. **Pre-Training**: The process of training the model on large amounts of text data before fine-tuning it for specific NLP tasks.

**Advantages:**

1. **Improved Semantic Understanding**: Captures the meaning of entire sentences, allowing for more accurate understanding of context.
2. **Increased Representation Capacity**: Learns higher-dimensional representations than traditional word embeddings.
3. **Efficient Computation**: Faster to compute than traditional NLP tasks that rely on word embeddings.

**Applications:**

1. **Text Classification**
2. **Sentiment Analysis**
3. **Question Answering**
4. **Named Entity Recognition**
5. **Machine Translation**

**Limitations:**

1. **Data Requirements**: Requires large amounts of text data for pre-training and fine-tuning.
2. **Computational Resources**: Demands significant computational resources to train and deploy the model.
3. **Interpretability**: Difficulty in interpreting the meaning behind the learned sentence embeddings.

**Related Topics:**

1. **Word Embeddings**
2. **Natural Language Processing (NLP)**
3. **Deep Learning**
4. **Text Analysis**
5. **Machine Learning**

**Future Directions:**

1. **Multimodal Learning**: Exploring ways to incorporate multimodal inputs, such as images or audio, into sentence transformer models.
2. **Explainability Techniques**: Developing methods to provide insights into the learned sentence embeddings and their relationships with the input text.
3. **Transfer Learning**: Investigating ways to leverage pre-trained sentence transformer models for a wider range of NLP tasks.

**Industry Applications:**

1. **Content Recommendation**
2. **Sentiment Analysis for Customer Feedback**
3. **Question Answering Systems**
4. **Named Entity Recognition for Information Retrieval**

Overall, sentence transformers have the potential to revolutionize the field of natural language processing by providing more accurate and nuanced understanding of text data. However, further research is needed to address the limitations and explore new applications in this exciting area.

### Experiment 4C: RunnableParallel + RunnablePassthrough (The RAG Pattern)

This is the most important LCEL composition — used in every RAG application.

```
Input (question)
  ├──→ "Retriever" → context (documents)
  └──→ RunnablePassthrough → question (preserved)
  ▼
Prompt (combines context + question) → Model → Parser → Answer
```

In [15]:
# Simulated RAG pattern — no vector DB needed, just the LCEL structure

# Simulate a retriever with a function
knowledge_base = {
    "lcel": "LCEL is LangChain's declarative composition framework using the pipe operator.",
    "runnable": "The Runnable protocol provides invoke(), stream(), and batch() on every component.",
    "passthrough": "RunnablePassthrough forwards input unchanged, assign() adds new keys.",
    "parallel": "RunnableParallel executes multiple branches simultaneously.",
}

def fake_retriever(query: str) -> str:
    """Simulate document retrieval by keyword matching."""
    query_lower = query.lower()
    matched = [doc for key, doc in knowledge_base.items() if key in query_lower]
    return " ".join(matched) if matched else "No relevant documents found."

# RAG chain using the dictionary shorthand pattern
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question based ONLY on this context:\n\n{context}\n\n"
     "If the context doesn't contain the answer, say so."),
    ("human", "{question}")
])

rag_chain = (
    {"context": RunnableLambda(lambda x: fake_retriever(x["question"])),
     "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"])}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test with questions
questions = [
    "What is LCEL?",
    "How does RunnableParallel work?",
    "What is the weather today?",  # Not in knowledge base
]

for q in questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    answer = rag_chain.invoke({"question": q})
    print(f"A: {answer}")


Q: What is LCEL?
A: LCEL stands for LangChain's declarative composition framework using the pipe operator.

Q: How does RunnableParallel work?
A: RunnableParallel is a programming model that allows you to execute multiple branches of code concurrently, improving overall performance and responsiveness.

The specific details about how RunnableParallel works are not provided in the original context, which only mentions:

"The Runnable protocol provides invoke(), stream(), and batch() on every component. RunnableParallel executes multiple branches simultaneously."

However, based on general knowledge of programming models, it is likely that RunnableParallel uses a combination of the following techniques to execute multiple branches concurrently:

1. **Thread Pooling**: It creates a pool of worker threads that can be reused to execute different branches of code.
2. **Async Programming**: It uses asynchronous programming techniques, such as coroutines or futures, to enable concurrent execut

---

## 5. LCEL Primitive 3: RunnableLambda — Custom Functions as Runnables

`RunnableLambda` wraps any Python function into a Runnable, making it composable in LCEL chains.

Use it for: data transformation, validation, formatting, external API calls, logging — anything that doesn't fit a built-in component.

### Experiment 5A: Basic RunnableLambda

In [16]:
# Wrap any function as a Runnable
def word_count(text: str) -> dict:
    """Count words and characters."""
    return {
        "text": text,
        "words": len(text.split()),
        "chars": len(text)
    }

counter = RunnableLambda(word_count)

# It supports the full Runnable interface
result = counter.invoke("LCEL makes chains composable and powerful")
print(f"invoke(): {result}")

# batch() works too
results = counter.batch(["Hello world", "LCEL is great", "Three word sentence"])
print(f"\nbatch(): {results}")

invoke(): {'text': 'LCEL makes chains composable and powerful', 'words': 6, 'chars': 41}

batch(): [{'text': 'Hello world', 'words': 2, 'chars': 11}, {'text': 'LCEL is great', 'words': 3, 'chars': 13}, {'text': 'Three word sentence', 'words': 3, 'chars': 19}]


### Experiment 5B: RunnableLambda in a Chain — Custom Post-Processing

In [17]:
# Custom functions as chain steps

def format_as_bullet_points(text: str) -> str:
    """Convert newline-separated text into bullet points."""
    lines = [line.strip() for line in text.strip().split("\n") if line.strip()]
    return "\n".join(f"  * {line}" for line in lines)

def add_header(text: str) -> str:
    """Add a formatted header."""
    return f"{'=' * 40}\n  LCEL CHEAT SHEET\n{'=' * 40}\n{text}\n{'=' * 40}"

# Chain: prompt → model → parse → format → header
formatted_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "List each item on its own line. No numbering."),
        ("human", "List the 5 LCEL primitives and what each does.")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(format_as_bullet_points)
    | RunnableLambda(add_header)
)

result = formatted_chain.invoke({})
print(result)

  LCEL CHEAT SHEET
  * The 5 LCE (Learning, Caching, and Exploitation) primitives are a set of fundamental building blocks for designing efficient and scalable deep learning models, particularly those used in computer vision and natural language processing tasks. Here are the 5 LCE primitives, along with their functions:
  * 1. **Learning**: This primitive involves training a model on a dataset, typically using an optimization algorithm such as stochastic gradient descent (SGD) or Adam. The goal of learning is to minimize the loss function between the predicted outputs and the actual labels.
  * 2. **Caching**: Caching refers to storing intermediate results or features learned during the learning process to avoid re-computing them multiple times. This can help reduce computational cost, memory usage, and improve performance by avoiding redundant computations.
  * 3. **Exploitation**: Exploitation involves using the cached information (features, weights, etc.) to make predictions or tak

### Experiment 5C: The @chain Decorator — Cleaner Syntax

In [18]:
from langchain_core.runnables import chain as chain_decorator

# The @chain decorator turns a function into a Runnable
# Cleaner than wrapping with RunnableLambda

@chain_decorator
def analyze_and_respond(input_dict: dict) -> str:
    """A custom Runnable that does multi-step processing."""
    question = input_dict["question"]

    # Step 1: Classify the question type
    classify_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "Classify this as 'technical' or 'general'. One word only."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )
    category = classify_chain.invoke({"q": question}).strip().lower()

    # Step 2: Answer based on classification
    if "technical" in category:
        style = "a precise software engineer with code examples"
    else:
        style = "a friendly teacher using simple analogies"

    answer_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You are {style}. Keep answers to 2-3 sentences."),
            ("human", "{q}")
        ])
        | llm | StrOutputParser()
    )

    answer = answer_chain.invoke({"style": style, "q": question})
    return f"[{category.upper()}] {answer}"

# Use it like any other Runnable
result = analyze_and_respond.invoke({"question": "How does the pipe operator work in LCEL?"})
print(result)

print()

result2 = analyze_and_respond.invoke({"question": "Why is the sky blue?"})
print(result2)

[IN LOGIC-BASED ENGLISH CORPUS (LCEC), THE PIPE OPERATOR (`|`) IS USED TO REPRESENT A CONNECTION OR ASSOCIATION BETWEEN TWO CONCEPTS.

HERE'S HOW IT WORKS:

* THE PIPE OPERATOR `|` IS A BINARY RELATION THAT CONNECTS TWO SETS OF WORDS, PHRASES, OR SENTENCES.
* IT INDICATES AN INCLUSIVE RELATIONSHIP BETWEEN THE TWO SETS, MEANING THAT EVERY ELEMENT IN ONE SET IS ALSO AN ELEMENT IN THE OTHER SET.

IN LCEC, THE PIPE OPERATOR IS USED TO REPRESENT SEMANTIC RELATIONSHIPS BETWEEN CONCEPTS. FOR EXAMPLE:

WORD `DOG` | ANIMAL

HERE, THE WORD "DOG" IS ASSOCIATED WITH THE CONCEPT OF "ANIMAL". THE PIPE OPERATOR INDICATES THAT "DOG" IS A SUBTYPE OR A SPECIFIC TYPE OF ANIMAL.

ANOTHER EXAMPLE:

PERSON | HUMAN BEING

IN THIS CASE, THE PIPE OPERATOR CONNECTS TWO SETS: "PERSON" AND "HUMAN BEING". THIS RELATIONSHIP IMPLIES THAT EVERY PERSON IS ALSO A HUMAN BEING.

THE PIPE OPERATOR HAS SEVERAL PROPERTIES:

* **INCLUSIVITY**: AS MENTIONED EARLIER, IT INDICATES AN INCLUSIVE RELATIONSHIP BETWEEN TWO SETS.
* *

---

## 6. LCEL Primitive 4: RunnableBranch — Conditional Routing

`RunnableBranch` routes input to **different Runnables based on conditions**. Like an `if/elif/else` for chains.

```
Input
  ▼
RunnableBranch
  ├── condition_1 is True → chain_a
  ├── condition_2 is True → chain_b
  └── default → chain_c
```

### Experiment 6A: Basic Conditional Routing

In [19]:
# Different chains for different input types

technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a precise engineer. Use technical terminology. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Use metaphors and vivid language. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise. 2 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

# Route based on keywords in the question
router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["code", "api", "function", "implement"]),
     technical_chain),
    (lambda x: any(w in x["question"].lower() for w in ["story", "imagine", "creative", "poem"]),
     creative_chain),
    general_chain  # default
)

# Test with different types of questions
test_questions = [
    "How do I implement a chain in code?",
    "Imagine LCEL as a story character.",
    "What is the weather like today?",
]

for q in test_questions:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    print(f"{'=' * 50}")
    result = router.invoke({"question": q})
    print(result)


Q: How do I implement a chain in code?
[TECHNICAL] Implementing a Chain Data Structure in Code

A chain is a data structure that consists of nodes linked together, where each node points to the next node in the sequence. Here's an example implementation of a chain in Python:

```python
class Node:
    """Represents a single node in the chain."""
    
    def __init__(self, value):
        """
        Initializes a new node with the given value.
        
        :param value: The value to store in the node.
        """
        self.value = value
        self.next = None

class Chain:
    """Represents a chain of nodes."""
    
    def __init__(self):
        """
        Initializes an empty chain.
        """
        self.head = None

    def append(self, value):
        """
        Appends a new node to the end of the chain with the given value.
        
        :param value: The value to store in the new node.
        """
        if not self.head:
            self.head = Node(value)


### Experiment 6B: LLM-Powered Routing (Classify then Route)

In [20]:
# Use the LLM itself to classify, then route to the right chain

# Step 1: Classify the question
classifier = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Classify this question into exactly ONE category: technical, creative, or general.\n"
         "Respond with the category name only. One word."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Use assign() to keep question AND add classification
classified_chain = RunnablePassthrough.assign(
    category=classifier
)

# Step 3: Route based on classification
smart_router = RunnableBranch(
    (lambda x: "technical" in x["category"].lower(), technical_chain),
    (lambda x: "creative" in x["category"].lower(), creative_chain),
    general_chain
)

# Full pipeline: classify → route → respond
smart_chain = classified_chain | smart_router

test_qs = [
    "How does the Runnable protocol handle async execution?",
    "Write me a poem about data pipelines.",
    "What should I have for lunch?",
]

for q in test_qs:
    print(f"\n{'=' * 50}")
    print(f"Q: {q}")
    # Show classification
    classified = classified_chain.invoke({"question": q})
    print(f"Category: {classified['category'].strip()}")
    print(f"{'=' * 50}")
    result = smart_router.invoke(classified)
    print(result)


Q: How does the Runnable protocol handle async execution?
Category: technical
[TECHNICAL] The `Runnable` interface in Java is not directly related to async execution, but rather it's a basic contract for objects that can be executed by a thread pool.

To execute a `Runnable` asynchronously, you typically use one of the following approaches:

1. **ExecutorService**: You submit your `Runnable` instance to an `ExecutorService`, which manages a pool of worker threads. The `ExecutorService` will schedule the execution of your `Runnable` instance when a thread is available in the pool.

```java
// Create an ExecutorService with 5 threads
ExecutorService executor = Executors.newFixedThreadPool(5);

// Submit your Runnable instance to the executor
executor.submit(new MyRunnable());
```

2. **Future**: You can use the `Future` class, which represents the result of an asynchronous operation. In this case, you create a `FutureTask` that wraps your `Runnable` instance.

```java
// Create a Future

---

## 7. LCEL Primitive 5: Fallbacks — Error Recovery

`.with_fallbacks()` wraps a Runnable with backup alternatives. If the primary fails, it automatically tries the next one.

```python
robust_model = primary_model.with_fallbacks([backup_model_1, backup_model_2])
```

### Experiment 7A: Fallback Chain

In [21]:
# Simulate a failing model with a function that always raises

def unreliable_model(input_text):
    """Simulates an LLM that fails."""
    raise ConnectionError("Model API is down!")

# Primary: the unreliable model
primary = RunnableLambda(unreliable_model)

# Fallback: our reliable Ollama model
fallback_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

# Combine with fallback
resilient_chain = primary.with_fallbacks([fallback_chain])

# The primary will fail, but the fallback catches it
result = resilient_chain.invoke({"question": "What is LCEL?"})

print(f"Primary failed but fallback succeeded!")
print(f"Result: {result}")

Primary failed but fallback succeeded!
Result: LCEL (Least Common Equivalent Language) is a language model that generates text in the style of a target language, using the most common equivalent words and phrases from another language.

LCEL was introduced by Google in 2020 as a way to generate human-like text in a specific language. It's based on the idea of finding the "least common equivalent" of a word or phrase between two languages, which means choosing the word that is most similar in both languages.

Here's how LCEL works:

1. **Language pair**: The model takes two languages as input: the source language (e.g., English) and the target language (e.g., Spanish).
2. **Word embedding**: The model uses a word embedding technique to represent each word in the source language as a vector in a high-dimensional space.
3. **Least common equivalent search**: For each word in the target language, the model searches for its most similar equivalent in the source language using the vector rep

### Experiment 7B: Retry Logic with `.with_retry()`

In [24]:
# .with_retry() adds automatic retries with exponential backoff

attempt_count = 0

def flaky_function(x):
    """Fails twice, then succeeds on the third attempt."""
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        print(f"  Attempt {attempt_count}: FAILED")
        raise ConnectionError(f"Transient error on attempt {attempt_count}")
    print(f"  Attempt {attempt_count}: SUCCESS")
    return f"Result after {attempt_count} attempts: {x}"

# Wrap with retry logic
retry_runnable = RunnableLambda(flaky_function).with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

print("Invoking flaky function with retry...")
result = retry_runnable.invoke("hello")
print(f"\nFinal result: {result}")

Invoking flaky function with retry...
  Attempt 1: FAILED
  Attempt 2: FAILED
  Attempt 3: SUCCESS

Final result: Result after 3 attempts: hello


### Experiment 7C: Combining Retry + Fallback (Maximum Resilience)

The most robust pattern:
```python
resilient = (
    primary.with_retry(stop_after_attempt=2)
    .with_fallbacks([
        fallback.with_retry(stop_after_attempt=2)
    ])
)
```

In [23]:
# Demonstrate the pattern conceptually

# In production with multiple providers, this would look like:
# resilient_llm = (
#     ChatOpenAI(model="gpt-4o").with_retry(stop_after_attempt=2)
#     .with_fallbacks([
#         ChatAnthropic(model="claude-3-5-sonnet").with_retry(stop_after_attempt=2),
#         ChatOllama(model="llama3.2:3b")  # local fallback, no retry needed
#     ])
# )

# With our local setup — retry the same model
resilient_llm = llm.with_retry(stop_after_attempt=3)

resilient_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Be concise. One sentence."),
        ("human", "{question}")
    ])
    | resilient_llm
    | StrOutputParser()
)

result = resilient_chain.invoke({"question": "What makes LCEL resilient?"})
print(result)

LCEL (Local Context Embedding Codebook) is a type of neural network-based embedding model that is designed to be resilient under various forms of data perturbation, such as label noise, semantic noise, and adversarial attacks.

Several factors contribute to the resilience of LCEL:

1. **Robustness to label noise**: LCEL uses a codebook-based approach, which allows it to effectively represent noisy labels by learning a set of representative codes that are not sensitive to individual label assignments.
2. **Temporal redundancy**: The model captures temporal relationships between consecutive frames in a video sequence, making it less vulnerable to attacks that rely on sudden changes in the input data.
3. **Spatial locality**: LCEL's embedding space is designed to capture local structures and patterns in the data, which helps to maintain its representation stability even when exposed to various types of noise or perturbations.
4. **Simple architecture**: The model has a relatively simple a

---

## 8. Streaming Deep Dive

When you call `.stream()` on an LCEL chain, every component streams its output to the next as tokens become available.

| Component | Streaming Behavior |
|-----------|-----------------------|
| Chat Model | Yields tokens as generated |
| StrOutputParser | Passes through each token immediately |
| JsonOutputParser | Accumulates until valid JSON, then yields |
| RunnablePassthrough | Passes through immediately |
| RunnableLambda | Depends on function (generators stream) |
| Retriever | Returns complete results (no partial streaming) |

### Experiment 8A: Streaming Through a Chain — Token Timing

In [31]:
# Compare: invoke (wait for all) vs stream (token by token)

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

question = {"question": "List 3 benefits of Langchain Expression Language in langchain. One sentence each."}

# --- invoke: wait for complete response ---
print("=" * 50)
print("invoke() — User waits for full response")
print("=" * 50)
start = time.time()
result = chain.invoke(question)
elapsed = time.time() - start
print(result)
print(f"\nTotal wait: {elapsed:.2f}s")

invoke() — User waits for full response
Here are three benefits of the Langchain Expression Language:

1. **Easier modeling of complex NLP tasks**: The Expression Language allows for more intuitive and declarative modeling of complex NLLs, making it easier to define and train models.
2. **Increased flexibility in model customization**: The language's syntax enables users to customize models with ease, allowing for rapid prototyping and iteration of different model architectures.
3. **Improved readability and maintainability**: The Expression Language promotes a more readable and maintainable codebase by abstracting away low-level implementation details, making it easier for developers to focus on the logic of their models.

Total wait: 2.67s


In [32]:
# --- stream: tokens arrive immediately ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)
start = time.time()
first_token_time = None
token_count = 0

for chunk in chain.stream(question):
    if first_token_time is None:
        first_token_time = time.time() - start
    print(chunk, end="", flush=True)
    token_count += 1

elapsed = time.time() - start
print(f"\n\nTime to first token: {first_token_time:.2f}s")
print(f"Total time: {elapsed:.2f}s")
print(f"Chunks received: {token_count}")
print("\nUsers perceive streaming as faster even when total time is similar!")

stream() — Tokens arrive as generated
Here are three benefits of Langchain's Expression Language:

1. **Flexible and Modular**: The Expression Language allows developers to create complex models by breaking them down into smaller, modular components that can be easily combined and reused.
2. **Improved Model Explainability**: By allowing users to define their own evaluation metrics and query interfaces, the Expression Language enables more transparent and interpretable model performance, making it easier to understand what a model is learning from its input data.
3. **Easier Model Maintenance and Updates**: The Expression Language provides a standardized way of defining models, making it easier for developers to update or modify existing models without affecting their underlying structure or functionality.

Time to first token: 0.13s
Total time: 2.53s
Chunks received: 134

Users perceive streaming as faster even when total time is similar!


---

## 9. Batch Processing Deep Dive

`.batch()` processes multiple inputs in **parallel**. Configurable with `max_concurrency`.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_concurrency` | None (unlimited) | Max parallel executions |
| `return_exceptions` | False | Return exceptions instead of raising |

### Experiment 9A: Batch with Timing Comparison

In [27]:
chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Answer in exactly one sentence."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
)

questions = [
    {"question": "What is LCEL?"},
    {"question": "What is a Runnable?"},
    {"question": "What is RunnablePassthrough?"},
    {"question": "What is RunnableParallel?"},
]

# --- Sequential (invoke one by one) ---
print("Sequential (invoke one by one):")
start = time.time()
sequential_results = [chain.invoke(q) for q in questions]
seq_time = time.time() - start
print(f"  Time: {seq_time:.2f}s")

# --- Batch (all at once) ---
print("\nBatch (parallel):")
start = time.time()
batch_results = chain.batch(questions)
batch_time = time.time() - start
print(f"  Time: {batch_time:.2f}s")

print(f"\nSpeedup: {seq_time/batch_time:.1f}x faster with batch!")

print("\nResults:")
for q, r in zip(questions, batch_results):
    print(f"  Q: {q['question']}")
    print(f"  A: {r}\n")

Sequential (invoke one by one):
  Time: 23.07s

Batch (parallel):
  Time: 27.05s

Speedup: 0.9x faster with batch!

Results:
  Q: What is LCEL?
  A: LCE (Local Contrastive Encoding) is a supervised learning method used to optimize the embedding space of a neural network, such as a sentence transformer model. It is particularly useful for tasks like text classification and clustering.

In traditional contrastive learning methods, similar samples are paired together in the training process to encourage their embeddings to be close, while dissimilar samples are paired together to discourage their embeddings from being too similar. However, this approach can lead to overfitting when the dataset size is limited or when there are many classes with few samples.

LCE addresses these challenges by introducing a new loss function that focuses on local contrastive learning. The idea is to find a balance between encouraging similarity and discouraging similarity, but in a more targeted way.

In LC

In [28]:
# Batch with max_concurrency — respect rate limits
print("Batch with max_concurrency=2:")
start = time.time()
results = chain.batch(questions, config={"max_concurrency": 2})
elapsed = time.time() - start
print(f"  Time: {elapsed:.2f}s (limited to 2 concurrent requests)")
print(f"  Results: {len(results)} answers")

Batch with max_concurrency=2:
  Time: 29.38s (limited to 2 concurrent requests)
  Results: 4 answers


---

## 10. Advanced Pattern: Sequential Processing with Context Accumulation

Build up context progressively — each step has access to everything produced by previous steps.

```
Input → Step 1 (summary) → Step 2 (key points using summary) → Step 3 (report using all above)
```

### Experiment 10A: Multi-Step Analysis with assign()

In [33]:
# Progressive context accumulation using assign()

# Step 1: Summarize the topic
summarize = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in 1-2 sentences."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 2: Extract key points (has access to topic + summary)
extract_points = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Given this summary: {summary}\n\n"
         "List 3 key points about the original topic. One line each."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Step 3: Generate a verdict (has access to topic + summary + key_points)
verdict = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Based on:\nSummary: {summary}\nKey Points: {key_points}\n\n"
         "Give a one-sentence verdict on the topic's importance."),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# Chain with progressive context accumulation
analysis_chain = (
    RunnablePassthrough.assign(summary=summarize)         # adds 'summary'
    | RunnablePassthrough.assign(key_points=extract_points)  # adds 'key_points'
    | RunnablePassthrough.assign(verdict=verdict)            # adds 'verdict'
)

result = analysis_chain.invoke({"topic": "LCEL (LangChain Expression Language)"})

print("PROGRESSIVE ANALYSIS")
print("=" * 50)
print(f"\nTopic: {result['topic']}")
print(f"\nSummary:\n  {result['summary']}")
print(f"\nKey Points:\n  {result['key_points']}")
print(f"\nVerdict:\n  {result['verdict']}")

PROGRESSIVE ANALYSIS

Topic: LCEL (LangChain Expression Language)

Summary:
  LCLE (LangChain Expression Language, pronounced "lecle") is a programming language designed specifically for natural language processing (NLP) tasks, particularly those involving text generation, summarization, and text manipulation.

**Overview**

LCLE is an extension of the LangChain ecosystem, which provides a set of libraries and tools for building NLP applications. LCLE adds support for defining custom expression languages that can be used to manipulate and generate text.

**Key Features**

1. **Expression Language**: LCLE introduces a new programming language called Expression Language (EXL) that allows you to define custom expressions using a simple syntax.
2. **Template-based Text Generation**: LCLE provides a template-based approach for generating text, which enables you to create complex templates and manipulate them dynamically.
3. **Text Manipulation**: LCLE includes a range of text manipulation t

---

## 11. LCEL Best Practices

| Practice | Description |
|----------|-------------|
| Keep chains simple | Start with `prompt \| model \| parser`, add complexity only when needed |
| Name your chains | Use `.with_config(run_name="name")` for easier debugging |
| Use `assign()` for context | Cleaner than manual dictionary construction |
| Prefer dict shorthand | `{"key": runnable}` over explicit `RunnableParallel` |
| Always add fallbacks | LLM APIs fail — plan for it with `.with_fallbacks()` |
| Use retry with backoff | `.with_retry()` handles transient errors gracefully |
| Set `max_concurrency` | Respect API rate limits when using `.batch()` |
| Stream for user-facing apps | Users perceive streaming as faster |
| Test with `invoke()` first | Debug synchronous execution before adding streaming |

### LCEL vs Imperative Code

| Aspect | LCEL (Declarative) | Imperative Code |
|--------|--------------------|-----------------|
| Readability | Excellent for linear/parallel flows | Better for complex branching |
| Streaming | Automatic, first-class | Manual implementation |
| Async | Built-in for every chain | Must rewrite with async/await |
| Batching | One method call | Custom threading |
| Tracing | Automatic (LangSmith) | Manual logging |
| Error handling | Declarative retry/fallback | Try/except blocks |
| Best for | 90% of LLM pipelines | Highly dynamic workflows |

> **Recommendation:** Use LCEL for 90% of your chains. Fall back to imperative code only for highly dynamic workflows — or consider **LangGraph** for those.

---

## 12. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Build your own LCEL chain
my_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}")
])

my_chain = my_prompt | llm | StrOutputParser()

# Try different modes
mode = "stream"  # Options: "invoke", "stream", "batch"

# ============================================================

if mode == "invoke":
    print("[INVOKE]\n")
    result = my_chain.invoke({"question": "What is the most important LCEL primitive and why?"})
    display(Markdown(result))

elif mode == "stream":
    print("[STREAM]\n")
    for chunk in my_chain.stream({"question": "What is the most important LCEL primitive and why?"}):
        print(chunk, end="", flush=True)
    print()

elif mode == "batch":
    print("[BATCH]\n")
    results = my_chain.batch([
        {"question": "What is RunnablePassthrough?"},
        {"question": "What is RunnableBranch?"},
        {"question": "What is RunnableLambda?"},
    ])
    for r in results:
        print(f"  {r}\n")

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LCEL** | Declarative composition with the pipe `\|` operator |
| **Runnable Protocol** | Universal interface: `invoke()`, `stream()`, `batch()` on every component |
| **Fundamental Pattern** | `prompt \| model \| parser` — the atom of every chain |
| **RunnablePassthrough** | Forwards data unchanged; `assign()` enriches with new keys |
| **RunnableParallel** | Executes branches simultaneously; dict shorthand `{"key": runnable}` |
| **RunnableLambda** | Wraps any Python function as a composable Runnable |
| **@chain decorator** | Cleaner syntax for custom Runnables |
| **RunnableBranch** | Conditional routing — if/elif/else for chains |
| **with_fallbacks()** | Automatic failover to backup models/chains |
| **with_retry()** | Automatic retries with exponential backoff |
| **Streaming** | Automatic through the entire chain — no extra code needed |
| **Batching** | `.batch()` with `max_concurrency` for parallel + rate-limited processing |
| **RAG Pattern** | `{"context": retriever, "question": passthrough} \| prompt \| model \| parser` |
| **Context Accumulation** | Chain `assign()` calls to progressively build up data |